# mini-qforest — 01. Exploration and classical baselines

**Goal of the project:** detect forest on Sentinel-2 satellite images and compare three families of models:
classical machine learning, graph neural networks (notebook 02) and hybrid quantum classifiers (notebook 03).

**This notebook:**
1. downloads the EuroSAT dataset (27,000 Sentinel-2 patches, 64×64 px, 10 land-cover classes);
2. explores the classes;
3. extracts interpretable features (colour, vegetation index, texture);
4. trains two baselines: a random forest on the features and a small CNN on the raw images;
5. saves the features, the train/val/test split and the CNN embeddings for notebooks 02 and 03.

Two tasks are used throughout: **forest vs rest** (binary) and **10 classes**.

> In Colab: *Runtime → Change runtime type → GPU* before running.

In [ ]:
# --- Setup: works in Google Colab (results saved to your Google Drive) or locally ---
import os, sys, time
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = "/content/drive/MyDrive/mini-qforest"
else:
    WORKDIR = os.path.abspath("..")
DATA_DIR = os.path.join(WORKDIR, "data")
RES_DIR = os.path.join(WORKDIR, "results")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)
SEED = 42
print("Working directory:", WORKDIR)

In [ ]:
# --- Helper: every model writes its score to results/results.csv (shared by the 3 notebooks) ---
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

RESULTS_CSV = os.path.join(RES_DIR, "results.csv")

def log_result(model, task, y_true, y_pred, n_params=None, train_time_s=None):
    binary = len(np.unique(y_true)) == 2
    row = {
        "model": model,
        "task": task,
        "accuracy": round(accuracy_score(y_true, y_pred), 4),
        "f1": round(f1_score(y_true, y_pred, average="binary" if binary else "macro"), 4),
        "n_params": n_params,
        "train_time_s": None if train_time_s is None else round(train_time_s, 1),
    }
    df = pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else pd.DataFrame(columns=list(row))
    df = df[~((df["model"] == model) & (df["task"] == task))]
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df.to_csv(RESULTS_CSV, index=False)
    print(row)
    return row

## 1. Download EuroSAT

`torchvision` downloads the RGB version of EuroSAT. If its server is unavailable, the cell falls back to a copy hosted on Hugging Face (TorchGeo).

In [ ]:
import numpy as np
import torch
from torchvision import datasets

def load_eurosat(root):
    try:
        return datasets.EuroSAT(root=root, download=True)
    except Exception as e:
        print("torchvision download failed, using the Hugging Face mirror:", e)
        zip_path = os.path.join(root, "EuroSAT.zip")
        if not os.path.exists(zip_path):
            os.system(f"wget -q -O {zip_path} https://huggingface.co/datasets/torchgeo/eurosat/resolve/main/EuroSAT.zip")
        os.system(f"unzip -q -o {zip_path} -d {root}")
        return datasets.ImageFolder(os.path.join(root, "2750"))

ds = load_eurosat(DATA_DIR)
classes = list(ds.classes)
FOREST = classes.index("Forest")
print(len(ds), "images —", classes)

In [ ]:
# Load every image into memory (about 330 MB, takes 1–2 minutes)
from tqdm.auto import tqdm
X_img = np.stack([np.asarray(ds[i][0].convert("RGB"), dtype=np.uint8) for i in tqdm(range(len(ds)))])
y10 = np.array(ds.targets)
ybin = (y10 == FOREST).astype(int)
print(X_img.shape, "| forest share:", ybin.mean().round(3))

## 2. Exploration

In [ ]:
import matplotlib.pyplot as plt

counts = np.bincount(y10, minlength=len(classes))
plt.figure(figsize=(9, 3.5))
plt.bar(classes, counts, color=["#2e7d32" if c == "Forest" else "#9e9e9e" for c in classes])
plt.xticks(rotation=45, ha="right"); plt.ylabel("images"); plt.title("EuroSAT: images per class")
plt.tight_layout(); plt.savefig(os.path.join(RES_DIR, "class_counts.png"), dpi=150); plt.show()

In [ ]:
# Five random examples per class
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(len(classes), 5, figsize=(7, 14))
for c, name in enumerate(classes):
    for j, i in enumerate(rng.choice(np.where(y10 == c)[0], 5, replace=False)):
        axes[c, j].imshow(X_img[i]); axes[c, j].axis("off")
    axes[c, 0].set_title(name, loc="left", fontsize=9)
plt.tight_layout(); plt.savefig(os.path.join(RES_DIR, "examples.png"), dpi=120); plt.show()

## 3. Interpretable features

For each image we compute 35 features:

- mean and standard deviation of the R, G, B channels (6);
- **Green Leaf Index** GLI = (2G − R − B) / (2G + R + B), a vegetation index computable from RGB, and the excess-green index ExG (3);
- texture: mean absolute brightness gradient along both axes (2);
- 8-bin colour histogram per channel (24).

*With the 13-band EuroSAT version, the true NDVI = (NIR − Red) / (NIR + Red) would be used instead of GLI.*

In [ ]:
FEATURE_NAMES = (
    [f"mean_{c}" for c in "RGB"] + [f"std_{c}" for c in "RGB"]
    + ["gli_mean", "gli_std", "exg_mean", "grad_x", "grad_y"]
    + [f"hist_{c}_{b}" for c in "RGB" for b in range(8)]
)

def extract_features(imgs, bins=8, chunk=2000):
    out = []
    for s in range(0, len(imgs), chunk):
        x = imgs[s:s + chunk].astype(np.float32) / 255.0        # (n, H, W, 3)
        n = len(x)
        R, G, B = x[..., 0], x[..., 1], x[..., 2]
        gli = (2 * G - R - B) / (2 * G + R + B + 1e-6)
        exg = 2 * G - R - B
        gray = x.mean(axis=-1)
        f = [
            x.mean(axis=(1, 2)), x.std(axis=(1, 2)),
            gli.mean(axis=(1, 2))[:, None], gli.std(axis=(1, 2))[:, None], exg.mean(axis=(1, 2))[:, None],
            np.abs(np.diff(gray, axis=1)).mean(axis=(1, 2))[:, None],
            np.abs(np.diff(gray, axis=2)).mean(axis=(1, 2))[:, None],
        ]
        for c in range(3):
            idx = np.clip((x[..., c] * bins).astype(int), 0, bins - 1).reshape(n, -1)
            f.append(np.stack([(idx == b).mean(axis=1) for b in range(bins)], axis=1))
        out.append(np.concatenate(f, axis=1))
    return np.concatenate(out).astype(np.float32)

X_feat = extract_features(X_img)
assert X_feat.shape[1] == len(FEATURE_NAMES)
print("features:", X_feat.shape)

In [ ]:
# Does the vegetation index separate forest from the rest?
gli = X_feat[:, FEATURE_NAMES.index("gli_mean")]
plt.figure(figsize=(7, 3.5))
plt.hist(gli[ybin == 0], bins=60, alpha=0.6, density=True, label="other classes", color="#9e9e9e")
plt.hist(gli[ybin == 1], bins=60, alpha=0.7, density=True, label="Forest", color="#2e7d32")
plt.xlabel("mean Green Leaf Index"); plt.ylabel("density"); plt.legend(); plt.title("Vegetation index by class")
plt.tight_layout(); plt.savefig(os.path.join(RES_DIR, "gli_forest.png"), dpi=150); plt.show()

## 4. One split for all notebooks

Stratified split 70 % train / 15 % validation / 15 % test, fixed seed. All models in the three notebooks are evaluated on the same test images, so their scores are comparable.

In [ ]:
from sklearn.model_selection import train_test_split

idx = np.arange(len(y10))
idx_train, idx_tmp = train_test_split(idx, test_size=0.30, stratify=y10, random_state=SEED)
idx_val, idx_test = train_test_split(idx_tmp, test_size=0.50, stratify=y10[idx_tmp], random_state=SEED)

np.savez(os.path.join(RES_DIR, "features.npz"), X=X_feat, y10=y10, ybin=ybin,
         classes=np.array(classes), feature_names=np.array(FEATURE_NAMES))
np.savez(os.path.join(RES_DIR, "split.npz"), train=idx_train, val=idx_val, test=idx_test)
print(len(idx_train), len(idx_val), len(idx_test))

## 5. Baseline 1 — Random forest on the features

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay

for task, y in [("forest_vs_rest", ybin), ("10_classes", y10)]:
    rf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED,
                                class_weight="balanced" if task == "forest_vs_rest" else None)
    t0 = time.time(); rf.fit(X_feat[idx_train], y[idx_train]); dt = time.time() - t0
    log_result("Random forest (35 features)", task, y[idx_test], rf.predict(X_feat[idx_test]), train_time_s=dt)

# rf is now the 10-class model
fig, ax = plt.subplots(figsize=(7, 7))
ConfusionMatrixDisplay.from_predictions(y10[idx_test], rf.predict(X_feat[idx_test]), display_labels=classes,
                                        xticks_rotation=60, normalize="true", values_format=".2f", ax=ax, colorbar=False)
plt.title("Random forest — 10 classes (row-normalised)"); plt.tight_layout()
plt.savefig(os.path.join(RES_DIR, "confusion_rf.png"), dpi=150); plt.show()

top = np.argsort(rf.feature_importances_)[::-1][:12]
plt.figure(figsize=(6, 4)); plt.barh([FEATURE_NAMES[i] for i in top][::-1], rf.feature_importances_[top][::-1])
plt.title("Most important features"); plt.tight_layout(); plt.show()

## 6. Baseline 2 — Small CNN on the raw images

Three convolution blocks, about 95,000 parameters, 12 epochs with random flips. The best epoch on the validation set is kept.

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

X_t = torch.from_numpy(X_img).permute(0, 3, 1, 2).float() / 255.0
mean = X_t[idx_train].mean(dim=(0, 2, 3), keepdim=True)
std = X_t[idx_train].std(dim=(0, 2, 3), keepdim=True)
X_t = (X_t - mean) / std
y_t = torch.from_numpy(y10).long()

def loader(ids, shuffle):
    ids = torch.from_numpy(ids)
    return DataLoader(TensorDataset(X_t[ids], y_t[ids]), batch_size=128, shuffle=shuffle)

class SmallCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128))
        self.pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(128, n_classes))
    def embed(self, x):
        return self.pool(self.features(x))
    def forward(self, x):
        return self.head(self.embed(x))

cnn = SmallCNN().to(device)
n_params_cnn = sum(p.numel() for p in cnn.parameters())
print("parameters:", n_params_cnn)

In [ ]:
@torch.no_grad()
def predict(model, ids):
    model.eval()
    return torch.cat([model(xb.to(device)).argmax(1).cpu() for xb, _ in loader(ids, False)]).numpy()

opt = torch.optim.Adam(cnn.parameters(), lr=1e-3)
train_loader = loader(idx_train, True)
best_val, best_state, history = 0.0, None, []
t0 = time.time()
for epoch in range(12):
    cnn.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        if torch.rand(1) < 0.5: xb = torch.flip(xb, dims=[3])
        if torch.rand(1) < 0.5: xb = torch.flip(xb, dims=[2])
        opt.zero_grad(); loss = nn.functional.cross_entropy(cnn(xb), yb); loss.backward(); opt.step()
    val_acc = (predict(cnn, idx_val) == y10[idx_val]).mean()
    history.append(val_acc)
    if val_acc > best_val:
        best_val, best_state = val_acc, {k: v.detach().clone() for k, v in cnn.state_dict().items()}
    print(f"epoch {epoch + 1:2d}  loss {loss.item():.3f}  val acc {val_acc:.3f}")
dt = time.time() - t0
cnn.load_state_dict(best_state)

pred10 = predict(cnn, idx_test)
log_result("Small CNN (raw images)", "10_classes", y10[idx_test], pred10, n_params_cnn, dt)
log_result("Small CNN (raw images)", "forest_vs_rest", ybin[idx_test], (pred10 == FOREST).astype(int), n_params_cnn, dt)

In [ ]:
# Save 128-dimensional CNN embeddings: an alternative node representation for notebook 02
@torch.no_grad()
def embed_all(model):
    model.eval()
    dl = DataLoader(TensorDataset(X_t), batch_size=512, shuffle=False)
    return torch.cat([model.embed(xb.to(device)).cpu() for (xb,) in dl]).numpy()

emb = embed_all(cnn)
np.save(os.path.join(RES_DIR, "cnn_embeddings.npy"), emb.astype(np.float32))
print("embeddings:", emb.shape)

In [ ]:
pd.read_csv(RESULTS_CSV)

## Summary

- Fill in here, in 3–4 sentences, what the baselines achieve and which classes are confused (see the confusion matrix).
- Next: **02_graph_gnn.ipynb** connects similar images in a graph and tests whether graph neural networks improve on the same test set.